In [1]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP AND DEPENDENCIES
# =============================================================================

from pathlib import Path
import json
import os
import sys
import time
import warnings
import subprocess
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import entropy
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, label_binarize

warnings.filterwarnings("ignore")

# Mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Google Colab not detected. Running in local mode.")

# Project directory structure
BASE_DIR = (
    Path("/content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS")
    if IN_COLAB
    else Path.cwd() / "Forensic_Ready_XAI_IDS"
)
INTERMEDIATE_DIR = BASE_DIR / "intermediate_outputs"
ARTICLE_DIR      = BASE_DIR / "article_assets"
DATA_DIR         = BASE_DIR / "data"

for folder in [BASE_DIR, INTERMEDIATE_DIR, ARTICLE_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RESULTS_JSON = INTERMEDIATE_DIR / "experiment_results.json"
RESULTS_CSV  = INTERMEDIATE_DIR / "experiment_log.csv"

print(f"Base directory      : {BASE_DIR}")
print(f"Data directory      : {DATA_DIR}")
print(f"Intermediate JSON   : {RESULTS_JSON}")
print(f"Experiment log CSV  : {RESULTS_CSV}")


Mounted at /content/drive
Base directory      : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS
Data directory      : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/data
Intermediate JSON   : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_results.json
Experiment log CSV  : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_log.csv


In [2]:
# =============================================================================
# SECTION 2: EXPERIMENT REGISTRY AND HELPER FUNCTIONS
# =============================================================================

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

experiment_results = {
    "run_id": RUN_ID,
    "title": (
        "Forensic-Ready Explainable Intrusion Detection with "
        "Temporal Graph Modeling and Counterfactual Cyber Defense"
    ),
    "created_at": datetime.now().isoformat(),
    "status_notes": [],
    "dataset": {},
    "preprocessing": {},
    "split": {},
    "model": {},
    "evaluation": {},
    "xai": {},
    "forensics": {},
    "counterfactuals": {},
}

# --- Serialization helpers ---

def to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, Path):
        return str(obj)
    return obj

def deep_convert(value):
    if isinstance(value, dict):
        return {k: deep_convert(v) for k, v in value.items()}
    if isinstance(value, list):
        return [deep_convert(v) for v in value]
    return to_serializable(value)

# --- Result management ---

def update_results(section, payload):
    if section not in experiment_results:
        experiment_results[section] = {}
    if isinstance(experiment_results[section], dict) and isinstance(payload, dict):
        experiment_results[section].update(deep_convert(payload))
    else:
        experiment_results[section] = deep_convert(payload)

def add_status_note(note):
    experiment_results["status_notes"].append({
        "timestamp": datetime.now().isoformat(),
        "note": str(note),
    })

def save_results_json():
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(deep_convert(experiment_results), f, indent=2, ensure_ascii=False)

def append_experiment_log(row_dict):
    row_df = pd.DataFrame([deep_convert(row_dict)])
    if RESULTS_CSV.exists():
        row_df.to_csv(RESULTS_CSV, mode="a", index=False, header=False)
    else:
        row_df.to_csv(RESULTS_CSV, mode="w", index=False, header=True)

# --- Article asset folders ---

FIGURES_DIR = ARTICLE_DIR / "figures"
TABLES_DIR  = ARTICLE_DIR / "tables"
REPORTS_DIR = ARTICLE_DIR / "reports"

for folder in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --- File saving helpers ---

def sanitize_filename(name):
    return str(name).strip().replace(" ", "_").replace("/", "_")

def save_dataframe(df, filename, index=False):
    output_path = TABLES_DIR / sanitize_filename(filename)
    df.to_csv(output_path, index=index)
    return output_path

def save_excel(df, filename, index=False):
    output_path = TABLES_DIR / sanitize_filename(filename)
    df.to_excel(output_path, index=index)
    return output_path

def save_text_report(filename, text):
    output_path = REPORTS_DIR / sanitize_filename(filename)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)
    return output_path

def save_numpy_array(filename, arr):
    output_path = INTERMEDIATE_DIR / sanitize_filename(filename)
    np.save(output_path, arr)
    return output_path

SAVED_ARTIFACTS = []

def register_artifact(path_obj, description):
    SAVED_ARTIFACTS.append({"path": str(path_obj), "description": description})
    return path_obj

add_status_note("Experiment registry initialized.")
save_results_json()

print("Experiment registry is ready.")
print(f"Figures directory   : {FIGURES_DIR}")
print(f"Tables directory    : {TABLES_DIR}")
print(f"Reports directory   : {REPORTS_DIR}")


Experiment registry is ready.
Figures directory   : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/figures
Tables directory    : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables
Reports directory   : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/reports


In [3]:
# =============================================================================
# SECTION 3: DATASET LOADING AND VALIDATION
# =============================================================================

DATASET_PATH = DATA_DIR / "ton-iot-train_test_network.csv"

# Candidate column names for automatic detection
LABEL_CANDIDATES  = ["type", "label", "attack", "class", "category"]
SRC_IP_CANDIDATES = ["src_ip", "srcip", "src-ip", "src"]
DST_IP_CANDIDATES = ["dst_ip", "dstip", "dst-ip", "dst", "dest_ip"]
TIME_CANDIDATES   = ["timestamp", "ts", "time", "datetime", "date"]

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        "Please verify the Google Drive path."
    )

df = pd.read_csv(DATASET_PATH)

original_columns = list(df.columns)
normalized_map   = {col.strip().lower(): col for col in df.columns}

def find_first_matching_column(candidates, normalized_column_map):
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized_column_map:
            return normalized_column_map[key]
    return None

# Detect label column
LABEL_COLUMN = find_first_matching_column(LABEL_CANDIDATES, normalized_map)
if LABEL_COLUMN is None:
    raise ValueError(
        "No valid label column found in the dataset. "
        f"Tried: {LABEL_CANDIDATES}. Available columns: {original_columns}"
    )

# Detect optional forensic columns
SRC_IP_COL = find_first_matching_column(SRC_IP_CANDIDATES, normalized_map)
DST_IP_COL = find_first_matching_column(DST_IP_CANDIDATES, normalized_map)
TIME_COL   = find_first_matching_column(TIME_CANDIDATES, normalized_map)

# Synthesize timestamp from cumulative duration if no timestamp column is found
timestamp_synthesized = False
if TIME_COL is None and "duration" in normalized_map:
    base_time = pd.Timestamp("2024-01-01 00:00:00")
    df["timestamp"] = base_time + pd.to_timedelta(
        df["duration"].fillna(0).cumsum(), unit="s"
    )
    TIME_COL = "timestamp"
    timestamp_synthesized = True

# Save dataset summary to results registry
dataset_summary = {
    "dataset_path": str(DATASET_PATH),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "columns": original_columns,
    "missing_values_total": int(df.isna().sum().sum()),
    "label_column": LABEL_COLUMN,
    "label_distribution": {
        str(k): int(v)
        for k, v in df[LABEL_COLUMN].value_counts(dropna=False).items()
    },
    "identifier_availability": {
        "src_ip_present": SRC_IP_COL is not None,
        "dst_ip_present": DST_IP_COL is not None,
        "timestamp_present": TIME_COL is not None,
        "src_ip_column": SRC_IP_COL,
        "dst_ip_column": DST_IP_COL,
        "timestamp_column": TIME_COL,
    },
}

update_results("dataset", dataset_summary)
add_status_note("Dataset loaded and validated.")
save_results_json()

# Dataset description for reproducibility record
dataset_description = {
    "name"   : "TON-IoT Network Dataset",
    "source" : "https://research.unsw.edu.au/projects/toniot-datasets",
    "file"   : "ton-iot-train_test_network.csv",
    "classes": {
        "normal"    : "Benign network traffic",
        "backdoor"  : "Unauthorized remote access",
        "ddos"      : "Distributed Denial of Service attack",
        "dos"       : "Denial of Service attack",
        "injection" : "SQL or command injection attack",
        "mitm"      : "Man-in-the-Middle attack",
        "password"  : "Password brute-force or credential attack",
        "ransomware": "Ransomware attack",
        "scanning"  : "Network scanning or reconnaissance",
        "xss"       : "Cross-Site Scripting attack",
    },
    "timestamp_note": (
        "No timestamp column was present in the original dataset. "
        "A synthetic timestamp was generated from cumulative duration "
        "to support temporal graph construction."
    ),
}

desc_path = INTERMEDIATE_DIR / "dataset_description.json"
with open(desc_path, "w", encoding="utf-8") as f:
    json.dump(dataset_description, f, indent=2, ensure_ascii=False)

print("Dataset loaded successfully.")
print(f"  Name               : {dataset_description['name']}")
print(f"  Source             : {dataset_description['source']}")
print(f"  Shape              : {df.shape}")
print(f"  Label column       : {LABEL_COLUMN}")
print(f"  SRC_IP_COL         : {SRC_IP_COL}")
print(f"  DST_IP_COL         : {DST_IP_COL}")
print(f"  TIME_COL           : {TIME_COL} {'(synthesized)' if timestamp_synthesized else ''}")
print(f"  Timestamp note     : {dataset_description['timestamp_note']}")
print("\nLabel distribution:")
print(pd.Series(dataset_summary["label_distribution"]))
print("\nClass descriptions:")
for cls, desc in dataset_description["classes"].items():
    print(f"  {cls:12s} : {desc}")

Dataset loaded successfully.
  Name               : TON-IoT Network Dataset
  Source             : https://research.unsw.edu.au/projects/toniot-datasets
  Shape              : (211043, 45)
  Label column       : type
  SRC_IP_COL         : src_ip
  DST_IP_COL         : dst_ip
  TIME_COL           : timestamp (synthesized)
  Timestamp note     : No timestamp column was present in the original dataset. A synthetic timestamp was generated from cumulative duration to support temporal graph construction.

Label distribution:
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
dtype: int64

Class descriptions:
  normal       : Benign network traffic
  backdoor     : Unauthorized remote access
  ddos         : Distributed Denial of Service attack
  dos          : Denial of Service attack
  injection    : SQL or command injection attack
  mitm     

In [4]:
# =============================================================================
# SECTION 4: DATA CLEANING
# =============================================================================

df_clean = df.copy()

# Remove exact duplicate rows
n_rows_before      = len(df_clean)
df_clean           = df_clean.drop_duplicates()
n_rows_after_dedup = len(df_clean)

# Replace infinite values with NaN before imputation
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Collect forensic identifier columns that are present after deduplication
identifier_cols = [
    c for c in [SRC_IP_COL, DST_IP_COL, TIME_COL]
    if c is not None and c in df_clean.columns
]

# Impute missing values per column type
# Categorical columns receive a constant "unknown" fill
# Numeric columns receive median imputation
for col in df_clean.columns:
    if col == LABEL_COLUMN:
        continue
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].fillna("unknown")
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

cleaning_summary = {
    "n_rows_before": int(n_rows_before),
    "n_rows_after_deduplication": int(n_rows_after_dedup),
    "duplicates_removed": int(n_rows_before - n_rows_after_dedup),
    "identifier_columns": identifier_cols,
    "remaining_missing_values_total": int(df_clean.isna().sum().sum()),
}

update_results("preprocessing", cleaning_summary)
add_status_note("Data cleaning completed.")
save_results_json()

print("Data cleaning completed.")
print(f"  Rows before         : {cleaning_summary['n_rows_before']}")
print(f"  Rows after dedup    : {cleaning_summary['n_rows_after_deduplication']}")
print(f"  Duplicates removed  : {cleaning_summary['duplicates_removed']}")
print(f"  Identifier columns  : {cleaning_summary['identifier_columns']}")
print(f"  Remaining missing   : {cleaning_summary['remaining_missing_values_total']}")


Data cleaning completed.
  Rows before         : 211043
  Rows after dedup    : 201566
  Duplicates removed  : 9477
  Identifier columns  : ['src_ip', 'dst_ip', 'timestamp']
  Remaining missing   : 0


In [5]:
# =============================================================================
# SECTION 5: FEATURE PREPARATION AND ENCODING
# =============================================================================

# Drop label column and any other target-like columns to prevent leakage
possible_target_like_cols = ["label", "type", "attack", "class", "category"]

feature_drop_cols = list(set(
    [LABEL_COLUMN]
    + identifier_cols
    + [c for c in possible_target_like_cols if c in df_clean.columns]
))

X_raw = df_clean.drop(columns=feature_drop_cols, errors="ignore").copy()
y_raw = df_clean[LABEL_COLUMN].astype(str).copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

numeric_cols     = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()

# Build preprocessing pipeline for numeric and categorical features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

feature_summary = {
    "dropped_columns": feature_drop_cols,
    "n_features_raw": int(X_raw.shape[1]),
    "n_numeric_features": len(numeric_cols),
    "n_categorical_features": len(categorical_cols),
    "numeric_columns": numeric_cols[:50],
    "categorical_columns": categorical_cols[:50],
    "target_classes": label_encoder.classes_.tolist(),
}

update_results("preprocessing", feature_summary)
add_status_note("Feature matrix and target vector prepared with leakage-safe column removal.")
save_results_json()

print("Feature preparation completed.")
print(f"  Dropped columns     : {feature_drop_cols}")
print(f"  Raw feature count   : {X_raw.shape[1]}")
print(f"  Numeric columns     : {len(numeric_cols)}")
print(f"  Categorical columns : {len(categorical_cols)}")
print(f"  Target classes      : {label_encoder.classes_.tolist()}")


Feature preparation completed.
  Dropped columns     : ['label', 'src_ip', 'dst_ip', 'timestamp', 'type']
  Raw feature count   : 40
  Numeric columns     : 16
  Categorical columns : 24
  Target classes      : ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']


In [6]:
# =============================================================================
# SECTION 6: TRAIN AND TEST SPLIT
# =============================================================================

# Preserve forensic context columns aligned with the feature split
context_df = (
    df_clean[identifier_cols].copy()
    if identifier_cols
    else pd.DataFrame(index=df_clean.index)
)

X_train_raw, X_test_raw, y_train, y_test, ctx_train, ctx_test = train_test_split(
    X_raw,
    y,
    context_df,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

split_summary = {
    "train_size": int(len(X_train_raw)),
    "test_size": int(len(X_test_raw)),
    "test_ratio": 0.30,
    "stratified": True,
    "random_state": 42,
}

update_results("split", split_summary)
add_status_note("Train/test split completed with stratification.")
save_results_json()

print("Train/test split completed.")
print(f"  Train size          : {split_summary['train_size']}")
print(f"  Test size           : {split_summary['test_size']}")
print(f"  Test ratio          : {split_summary['test_ratio']}")
print(f"  Stratified          : {split_summary['stratified']}")


Train/test split completed.
  Train size          : 141096
  Test size           : 60470
  Test ratio          : 0.3
  Stratified          : True


In [7]:
# =============================================================================
# SECTION 7: MODEL TRAINING AND DIAGNOSTICS
# =============================================================================

t0 = time.time()

# Initialize Random Forest classifier with balanced class weights
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.3,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", clf),
])

print("Model training started.")
t_step = time.time()
model.fit(X_train_raw, y_train)
train_time = time.time() - t_step
print(f"  Training completed in {train_time:.2f} seconds.")

# Extract trained pipeline components
trained_preprocessor = model.named_steps["preprocessor"]
trained_classifier   = model.named_steps["classifier"]

# Recover transformed feature names
try:
    feature_names = trained_preprocessor.get_feature_names_out()
    n_features = len(feature_names)
except Exception as e:
    feature_names = None
    n_features = None
    print(f"  Could not recover feature names: {repr(e)}")

# Tree-level diagnostics
n_trees    = len(trained_classifier.estimators_)
tree_depths = [e.tree_.max_depth for e in trained_classifier.estimators_]
tree_nodes  = [e.tree_.node_count  for e in trained_classifier.estimators_]

avg_depth         = float(np.mean(tree_depths))
max_depth_observed = int(np.max(tree_depths))
min_depth_observed = int(np.min(tree_depths))
avg_nodes          = float(np.mean(tree_nodes))
max_nodes          = int(np.max(tree_nodes))
min_nodes          = int(np.min(tree_nodes))

# Training accuracy
train_accuracy = model.score(X_train_raw, y_train)

# Top native feature importances
feature_importances = trained_classifier.feature_importances_
top_k = min(15, len(feature_importances))

if feature_names is not None:
    top_indices  = np.argsort(feature_importances)[::-1][:top_k]
    top_features = [
        {"feature": str(feature_names[i]), "importance": float(feature_importances[i])}
        for i in top_indices
    ]
else:
    top_features = []

# Overfitting heuristic notes
diagnostic_notes = []
if train_accuracy >= 0.999:
    diagnostic_notes.append(
        "Training accuracy is extremely high; monitor for overfitting against test metrics."
    )
if avg_depth >= 18:
    diagnostic_notes.append(
        "Average tree depth is high; this may increase variance and reduce interpretability."
    )
if n_features is not None and n_features >= 800:
    diagnostic_notes.append(
        "Transformed feature space is large; SHAP and LIME computations may be slower."
    )
if not diagnostic_notes:
    diagnostic_notes.append("No diagnostic flags detected at training stage.")

model_summary = {
    "model_name": "RandomForestClassifier",
    "hyperparameters": {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 0.3,
        "class_weight": "balanced_subsample",
        "n_jobs": -1,
        "random_state": 42,
    },
    "training_time_sec": float(train_time),
    "n_transformed_features": int(n_features) if n_features is not None else None,
    "tree_depth": {
        "avg_depth": avg_depth,
        "max_depth": max_depth_observed,
        "min_depth": min_depth_observed,
    },
    "tree_nodes": {
        "avg_nodes": avg_nodes,
        "max_nodes": max_nodes,
        "min_nodes": min_nodes,
    },
    "training_accuracy": float(train_accuracy),
    "top_feature_importances": top_features,
    "diagnostic_notes": diagnostic_notes,
}

update_results("model", model_summary)
add_status_note("Random Forest baseline model trained with diagnostics.")
save_results_json()

print(f"\nModel summary:")
print(f"  Model               : RandomForestClassifier")
print(f"  Trees               : {n_trees}")
print(f"  Avg depth           : {avg_depth:.2f}")
print(f"  Max depth           : {max_depth_observed}")
print(f"  Min depth           : {min_depth_observed}")
print(f"  Avg nodes           : {avg_nodes:.2f}")
print(f"  Transformed features: {n_features}")
print(f"  Training accuracy   : {train_accuracy:.4f}")
print(f"  Training time (sec) : {train_time:.2f}")

if top_features:
    print("\nTop feature importances:")
    for f in top_features[:10]:
        print(f"  {f['feature']} ({f['importance']:.6f})")

print("\nDiagnostic notes:")
for note in diagnostic_notes:
    print(f"  - {note}")

print(f"\nTotal elapsed time  : {time.time() - t0:.2f} seconds")


Model training started.
  Training completed in 213.55 seconds.

Model summary:
  Model               : RandomForestClassifier
  Trees               : 200
  Avg depth           : 93.62
  Max depth           : 124
  Min depth           : 52
  Avg nodes           : 3723.69
  Transformed features: 887
  Training accuracy   : 0.9999
  Training time (sec) : 213.55

Top feature importances:
  num__src_ip_bytes (0.164407)
  num__dst_port (0.114567)
  num__src_port (0.103032)
  num__duration (0.082915)
  num__src_bytes (0.066532)
  num__dst_ip_bytes (0.064701)
  cat__conn_state_REJ (0.053016)
  num__src_pkts (0.049619)
  num__dst_bytes (0.038033)
  cat__conn_state_S0 (0.033728)

Diagnostic notes:
  - Training accuracy is extremely high; monitor for overfitting against test metrics.
  - Average tree depth is high; this may increase variance and reduce interpretability.
  - Transformed feature space is large; SHAP and LIME computations may be slower.

Total elapsed time  : 217.73 seconds


In [8]:
# =============================================================================
# SECTION 8: MODEL EVALUATION AND ROC-AUC
# =============================================================================

y_pred  = model.predict(X_test_raw)
y_proba = model.predict_proba(X_test_raw)

# Core classification metrics
acc = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

cm          = confusion_matrix(y_test, y_pred)
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)

# ROC-AUC (one-vs-rest, macro average)
classes_numeric = np.arange(len(label_encoder.classes_))
y_test_bin      = label_binarize(y_test, classes=classes_numeric)

try:
    roc_auc_macro_ovr = roc_auc_score(
        y_test_bin, y_proba, average="macro", multi_class="ovr"
    )
except Exception:
    roc_auc_macro_ovr = None

evaluation_summary = {
    "accuracy": float(acc),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "roc_auc_macro_ovr": roc_auc_macro_ovr,
    "confusion_matrix": cm.tolist(),
    "class_order": label_encoder.classes_.tolist(),
    "classification_report": report_dict,
}

update_results("evaluation", evaluation_summary)

quality_note = (
    f"Macro-F1 = {f1_macro:.4f}. "
    + ("Promising baseline." if f1_macro >= 0.90 else "Needs improvement.")
)
add_status_note(quality_note)
save_results_json()

append_experiment_log({
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(),
    "model_name": "RandomForestClassifier",
    "accuracy": float(acc),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "roc_auc_macro_ovr": roc_auc_macro_ovr,
    "n_train": len(X_train_raw),
    "n_test": len(X_test_raw),
})

print("Evaluation completed.")
print(f"  Accuracy            : {acc:.4f}")
print(f"  Macro precision     : {precision_macro:.4f}")
print(f"  Macro recall        : {recall_macro:.4f}")
print(f"  Macro F1            : {f1_macro:.4f}")
print(f"  ROC-AUC macro OVR   : {roc_auc_macro_ovr}")
print("\nConfusion matrix:")
print(cm)


Evaluation completed.
  Accuracy            : 0.9890
  Macro precision     : 0.9642
  Macro recall        : 0.9740
  Macro F1            : 0.9687
  ROC-AUC macro OVR   : 0.9978651978569155

Confusion matrix:
[[ 5998     0     1     1     0     0     0     0     0     0]
 [    0  5896     3    42     3     8     2     0     0    46]
 [    0     0  5900     3    43     6     5     0    41     2]
 [    0    40     4  5810     8     3     3     0    17   113]
 [    0     2     5    10   263     7    15     0    10     1]
 [    0     6     4     2     6 13712     0     2     0     6]
 [    0     1     4     5    24     1  5955     0     5     5]
 [    0     0     0     0     0     0     0  4421     0     0]
 [    0     0    27     7    11     1     1     0  5953     0]
 [    0    28     0    77     0     0     0     0     0  5895]]


In [9]:
# =============================================================================
# SECTION 9: XAI DEPENDENCY CHECK
# =============================================================================

def ensure_package(pkg_name, import_name=None):
    if import_name is None:
        import_name = pkg_name.replace("-", "_")
    try:
        __import__(import_name)
        print(f"  {pkg_name} is available.")
    except ImportError:
        print(f"  Installing {pkg_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])
        print(f"  {pkg_name} installed.")

print("Checking XAI dependencies:")
ensure_package("lime")
ensure_package("shap")
ensure_package("dice-ml", "dice_ml")
print("All XAI dependencies are ready.")


Checking XAI dependencies:
  Installing lime...
  lime installed.
  shap is available.
  Installing dice-ml...
  dice-ml installed.
All XAI dependencies are ready.


In [10]:
# =============================================================================
# SECTION 9.1: LIBRARY VERSIONS AND REPRODUCIBILITY RECORD
# =============================================================================

import sklearn
import shap
import lime
import dice_ml
import networkx
import scipy
import importlib.metadata

def get_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except Exception:
        return "unknown"

versions = {
    "python"    : sys.version.split()[0],
    "numpy"     : np.__version__,
    "pandas"    : pd.__version__,
    "sklearn"   : sklearn.__version__,
    "shap"      : shap.__version__,
    "lime"      : get_version("lime"),
    "dice_ml"   : get_version("dice-ml"),
    "networkx"  : networkx.__version__,
    "scipy"     : scipy.__version__,
    "matplotlib": plt.matplotlib.__version__,
}

print("Library versions for reproducibility:")
for lib, ver in versions.items():
    print(f"  {lib:12s} : {ver}")

import json

versions_path = INTERMEDIATE_DIR / "library_versions.json"
with open(versions_path, "w", encoding="utf-8") as f:
    json.dump(versions, f, indent=2)

print(f"\nVersions saved to: {versions_path}")


Library versions for reproducibility:
  python       : 3.12.13
  numpy        : 2.0.2
  pandas       : 2.2.2
  sklearn      : 1.6.1
  shap         : 0.51.0
  lime         : 0.2.0.1
  dice_ml      : 0.12
  networkx     : 3.6.1
  scipy        : 1.16.3
  matplotlib   : 3.10.0

Versions saved to: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/library_versions.json


In [11]:
# =============================================================================
# SECTION 10: GLOBAL EXPLAINABILITY (SHAP WITH PERMUTATION FALLBACK)
# =============================================================================

import shap

t0 = time.time()

# Transform test data using the fitted preprocessor
X_test_transformed = preprocessor.transform(X_test_raw)
feature_names      = preprocessor.get_feature_names_out()

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

X_test_transformed = np.asarray(X_test_transformed, dtype=np.float32)

sample_n      = min(500, X_test_transformed.shape[0])
X_eval_sample = X_test_transformed[:sample_n]
y_eval_sample = np.asarray(y_test)[:sample_n]

explainability_summary = {"method_selected": None, "top_features": None}
use_shap               = False
shap_importance_df     = None

# Attempt SHAP TreeExplainer on a small sample for efficiency
try:
    shap_sample_n  = min(3, X_test_transformed.shape[0])
    X_shap_sample  = X_test_transformed[:shap_sample_n]
    explainer      = shap.TreeExplainer(trained_classifier)
    shap_values_raw = explainer.shap_values(X_shap_sample, check_additivity=False)

    if isinstance(shap_values_raw, list):
        shap_array    = np.stack(shap_values_raw, axis=0)
        mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 1))
    elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
        if shap_values_raw.shape[0] == X_shap_sample.shape[0]:
            mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=(0, 2))
        else:
            mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=(0, 1))
    else:
        mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=0)

    mean_abs_shap = mean_abs_shap / mean_abs_shap.sum()

    shap_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": mean_abs_shap,
    }).sort_values("importance", ascending=False)

    max_importance = float(shap_importance_df["importance"].max())

    if np.isfinite(max_importance) and not np.isnan(max_importance):
        use_shap = True
        explainability_summary["method_selected"] = "shap"
        print("SHAP explainability completed.")
    else:
        print(f"SHAP values are unstable (max={max_importance:.4e}). Falling back to permutation importance.")

except Exception as e:
    print(f"SHAP failed: {repr(e)}. Falling back to permutation importance.")

# Fallback: permutation importance
if not use_shap:
    print("Computing permutation importance...")
    perm_result = permutation_importance(
        trained_classifier,
        X_eval_sample,
        y_eval_sample,
        n_repeats=3,
        random_state=42,
        n_jobs=-1,
    )
    shap_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": perm_result.importances_mean,
    }).sort_values("importance", ascending=False)
    explainability_summary["method_selected"] = "permutation_importance"
    print("Permutation importance completed.")

top_k        = min(20, len(shap_importance_df))
top_features = shap_importance_df.head(top_k).to_dict(orient="records")

explainability_summary["top_features"] = [
    {"feature": str(row["feature"]), "importance": float(row["importance"])}
    for row in top_features
]

update_results("global_explainability", explainability_summary)
add_status_note(f"Global explainability completed using {explainability_summary['method_selected']}.")
save_results_json()

print(f"  Method used         : {explainability_summary['method_selected']}")
print(f"  Total time          : {time.time() - t0:.2f} seconds")
print("\nTop features:")
print(shap_importance_df.head(10).to_string(index=False))


SHAP explainability completed.
  Method used         : shap
  Total time          : 7.25 seconds

Top features:
           feature  importance
 num__src_ip_bytes    0.094139
 num__dst_ip_bytes    0.093340
     num__duration    0.092651
 cat__service_http    0.087260
     num__dst_port    0.080493
    cat__service_-    0.067657
cat__conn_state_SH    0.063528
     num__src_port    0.051968
    num__dst_bytes    0.051067
    num__dns_rcode    0.051047


In [12]:
# =============================================================================
# SECTION 11: LOCAL EXPLAINABILITY (LIME)
# =============================================================================

from lime.lime_tabular import LimeTabularExplainer

t_start = time.time()

X_train_lime_df = X_train_raw.copy()
X_test_lime_df  = X_test_raw.copy()

feature_names_lime = list(X_train_lime_df.columns)
class_names_lime   = list(label_encoder.classes_)

# Encode categorical columns as integers for LIME
categorical_cols_lime = X_train_lime_df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()
numeric_cols_lime = [c for c in feature_names_lime if c not in categorical_cols_lime]

categorical_indices = []
int_to_category     = {}
categorical_names   = {}

for col in categorical_cols_lime:
    train_vals = X_train_lime_df[col].astype(str).fillna("MISSING")
    test_vals  = X_test_lime_df[col].astype(str).fillna("MISSING")

    categories = sorted(train_vals.unique().tolist())
    mapping    = {cat: idx for idx, cat in enumerate(categories)}
    reverse    = {idx: cat for cat, idx in mapping.items()}

    X_train_lime_df[col] = train_vals.map(mapping).astype(int)
    X_test_lime_df[col]  = test_vals.map(lambda v: mapping.get(v, -1)).astype(int)

    int_to_category[col] = reverse
    col_idx = feature_names_lime.index(col)
    categorical_indices.append(col_idx)
    categorical_names[col_idx] = categories + ["<UNSEEN>"]

# Impute numeric columns
numeric_fill_values = {}
for col in numeric_cols_lime:
    X_train_lime_df[col] = pd.to_numeric(X_train_lime_df[col], errors="coerce")
    X_test_lime_df[col]  = pd.to_numeric(X_test_lime_df[col],  errors="coerce")
    fill_value = X_train_lime_df[col].median()
    if pd.isna(fill_value):
        fill_value = 0.0
    numeric_fill_values[col] = float(fill_value)
    X_train_lime_df[col] = X_train_lime_df[col].fillna(fill_value)
    X_test_lime_df[col]  = X_test_lime_df[col].fillna(fill_value)

X_train_lime_np = X_train_lime_df[feature_names_lime].to_numpy(dtype=np.float64)
X_test_lime_np  = X_test_lime_df[feature_names_lime].to_numpy(dtype=np.float64)

# Prediction function for LIME: decodes integer-encoded categoricals before scoring
def predict_fn_lime(x):
    x_df = pd.DataFrame(x, columns=feature_names_lime)
    for col in categorical_cols_lime:
        def decode(v):
            try:
                iv = int(round(float(v)))
            except Exception:
                return "MISSING"
            return int_to_category[col].get(iv, "MISSING") if iv != -1 else "MISSING"
        x_df[col] = x_df[col].apply(decode)
    for col in numeric_cols_lime:
        x_df[col] = pd.to_numeric(x_df[col], errors="coerce").fillna(
            numeric_fill_values[col]
        )
    return model.predict_proba(x_df)

# Initialize LIME explainer on a random subsample of training data
lime_train_n = min(5000, len(X_train_lime_np))
rng          = np.random.default_rng(42)
lime_idx     = rng.choice(len(X_train_lime_np), size=lime_train_n, replace=False)

explainer_lime = LimeTabularExplainer(
    training_data=X_train_lime_np[lime_idx],
    feature_names=feature_names_lime,
    class_names=class_names_lime,
    categorical_features=categorical_indices,
    categorical_names=categorical_names,
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)

# Explain the first test instance
idx_lime           = 0
instance_lime      = X_test_lime_np[idx_lime]
pred_probs_lime    = predict_fn_lime(instance_lime.reshape(1, -1))[0]
pred_class_idx     = int(np.argmax(pred_probs_lime))
pred_class_name    = class_names_lime[pred_class_idx]
true_class_name    = class_names_lime[int(y_test[idx_lime])]

exp_lime           = explainer_lime.explain_instance(
    instance_lime, predict_fn_lime, labels=[pred_class_idx], num_features=10
)
lime_exp_list = exp_lime.as_list(label=pred_class_idx)

# Save LIME results
update_results("xai", {
    "lime_local": {
        "instance_index": int(idx_lime),
        "predicted_class_index": pred_class_idx,
        "predicted_class_name": pred_class_name,
        "true_class_name": true_class_name,
        "top_features": [
            {"feature": str(f), "weight": float(w)} for f, w in lime_exp_list
        ],
    }
})
add_status_note("LIME local explanation completed.")
save_results_json()

print("LIME local explanation completed.")
print(f"  Predicted class     : {pred_class_name}")
print(f"  True class          : {true_class_name}")
print(f"  Total time          : {time.time() - t_start:.2f} seconds")
print("\nTop LIME features:")
for f, w in lime_exp_list:
    print(f"  {f}: {w:.4f}")


LIME local explanation completed.
  Predicted class     : ransomware
  True class          : ransomware
  Total time          : 2.54 seconds

Top LIME features:
  dst_port > 445.00: 0.0906
  duration <= 0.00: 0.0833
  conn_state=OTH: 0.0456
  http_response_body_len <= 0.00: 0.0259
  dst_pkts <= 0.00: 0.0222
  http_uri=-: 0.0212
  http_version=-: 0.0187
  weird_name=-: 0.0184
  ssl_resumed=-: 0.0162
  http_status_code <= 0.00: 0.0089


In [13]:
# =============================================================================
# SECTION 12: FORENSIC PREDICTION TABLE
# =============================================================================

# Reconstruct human-readable label strings for test predictions
true_labels      = [label_encoder.classes_[i] for i in y_test]
predicted_labels = [label_encoder.classes_[i] for i in y_pred]

# Build forensic table from test context (src/dst IPs and timestamps)
forensic_table = (
    ctx_test.copy().reset_index(drop=True)
    if not ctx_test.empty
    else pd.DataFrame(index=X_test_raw.index)
)

forensic_table["true_label"]            = true_labels
forensic_table["predicted_label"]       = predicted_labels
forensic_table["predicted_confidence"]  = y_proba.max(axis=1)

# Shannon entropy of the predicted probability distribution
entropy_scores = entropy(y_proba.T)

# Risk relative to the normal (benign) class
if "normal" in label_encoder.classes_:
    normal_idx   = list(label_encoder.classes_).index("normal")
    prob_normal  = y_proba[:, normal_idx]
    risk_vs_normal = 1.0 - prob_normal
    forensic_table["normal_class_probability"] = prob_normal
else:
    prob_normal    = np.zeros(len(y_proba))
    risk_vs_normal = 1.0 - y_proba.max(axis=1)
    forensic_table["normal_class_probability"] = np.nan

# Hybrid suspicion score: combines entropy (uncertainty) and risk vs normal
forensic_table["entropy_score"]     = entropy_scores
forensic_table["risk_vs_normal"]    = risk_vs_normal
forensic_table["suspicion_score"]   = 0.5 * entropy_scores + 0.5 * risk_vs_normal
forensic_table["is_misclassified"]  = (
    forensic_table["true_label"] != forensic_table["predicted_label"]
)

forensic_summary = {
    "n_rows": int(len(forensic_table)),
    "mean_predicted_confidence": float(forensic_table["predicted_confidence"].mean()),
    "mean_entropy_score": float(forensic_table["entropy_score"].mean()),
    "mean_risk_vs_normal": float(forensic_table["risk_vs_normal"].mean()),
    "mean_suspicion_score": float(forensic_table["suspicion_score"].mean()),
    "misclassification_count": int(forensic_table["is_misclassified"].sum()),
}

update_results("forensics", {"prediction_table_summary": forensic_summary})
add_status_note("Forensic prediction table constructed with hybrid suspicion scoring.")
save_results_json()

print("Forensic prediction table constructed.")
print(f"  Rows                : {forensic_summary['n_rows']}")
print(f"  Mean confidence     : {forensic_summary['mean_predicted_confidence']:.4f}")
print(f"  Mean entropy        : {forensic_summary['mean_entropy_score']:.4f}")
print(f"  Mean risk vs normal : {forensic_summary['mean_risk_vs_normal']:.4f}")
print(f"  Mean suspicion score: {forensic_summary['mean_suspicion_score']:.4f}")
print(f"  Misclassifications  : {forensic_summary['misclassification_count']}")
print("\nSample rows:")
print(forensic_table.head())


Forensic prediction table constructed.
  Rows                : 60470
  Mean confidence     : 0.9892
  Mean entropy        : 0.0289
  Mean risk vs normal : 0.7729
  Mean suspicion score: 0.4009
  Misclassifications  : 667

Sample rows:
          src_ip         dst_ip                     timestamp  true_label  \
0  192.168.1.193   192.168.1.37 2024-01-16 18:50:21.222205017  ransomware   
1  192.168.1.152  192.168.1.152 2024-01-15 06:32:43.060226186      normal   
2  192.168.1.180  192.168.1.190 2024-01-16 10:50:01.279935131      normal   
3  192.168.1.103    224.0.0.252 2024-01-15 15:31:25.666484177      normal   
4  192.168.1.152  192.168.1.190 2024-01-15 08:38:41.717384186      normal   

  predicted_label  predicted_confidence  normal_class_probability  \
0      ransomware                   1.0                       0.0   
1          normal                   1.0                       1.0   
2          normal                   1.0                       1.0   
3          normal         

In [14]:
# =============================================================================
# SECTION 13: TEMPORAL GRAPH CONSTRUCTION
# =============================================================================

G = nx.DiGraph()

graph_available = all(
    col in forensic_table.columns for col in [SRC_IP_COL, DST_IP_COL]
) if (SRC_IP_COL and DST_IP_COL) else False

if graph_available:
    for _, row in forensic_table.iterrows():
        src = row[SRC_IP_COL]
        dst = row[DST_IP_COL]
        if pd.isna(src) or pd.isna(dst):
            continue
        suspicion_value = float(row["suspicion_score"])
        if G.has_edge(src, dst):
            G[src][dst]["count"]         += 1
            G[src][dst]["suspicion_sum"] += suspicion_value
        else:
            G.add_edge(src, dst, count=1, suspicion_sum=suspicion_value)

    # Normalize edge suspicion by connection frequency
    for u, v, data in G.edges(data=True):
        data["suspicion_avg"] = data["suspicion_sum"] / max(data["count"], 1)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
else:
    n_nodes, n_edges = 0, 0

update_results("forensics", {
    "graph_available": graph_available,
    "graph_n_nodes": int(n_nodes),
    "graph_n_edges": int(n_edges),
})
add_status_note(
    "Temporal interaction graph constructed with normalized edge suspicion."
    if graph_available
    else "Graph construction skipped: src/dst identifier columns unavailable."
)
save_results_json()

print("Temporal graph construction completed.")
print(f"  Graph available     : {graph_available}")
print(f"  Nodes               : {n_nodes}")
print(f"  Edges               : {n_edges}")


Temporal graph construction completed.
  Graph available     : True
  Nodes               : 505
  Edges               : 667


In [15]:
# =============================================================================
# SECTION 14: SUSPICIOUS NODE RANKING
# =============================================================================

node_ranking = []

if graph_available and G.number_of_nodes() > 0:
    pagerank_scores = (
        nx.pagerank(G, alpha=0.85)
        if G.number_of_edges() > 0
        else {n: 0.0 for n in G.nodes()}
    )

    for node in G.nodes():
        out_suspicion = sum(
            G[node][nbr].get("suspicion_avg", 0.0) for nbr in G.successors(node)
        )
        in_suspicion = sum(
            G[pred][node].get("suspicion_avg", 0.0) for pred in G.predecessors(node)
        )
        out_degree = G.out_degree(node)
        in_degree  = G.in_degree(node)
        pagerank   = pagerank_scores.get(node, 0.0)

        # Composite risk score weighted by graph signals
        risk_score = (
            0.4 * out_suspicion
            + 0.3 * in_suspicion
            + 0.2 * pagerank
            + 0.1 * (out_degree + in_degree)
        )

        node_ranking.append({
            "node": node,
            "out_suspicion": float(out_suspicion),
            "in_suspicion": float(in_suspicion),
            "out_degree": int(out_degree),
            "in_degree": int(in_degree),
            "pagerank": float(pagerank),
            "risk_score": float(risk_score),
        })

    node_ranking = sorted(node_ranking, key=lambda x: x["risk_score"], reverse=True)
    top_nodes    = node_ranking[:20]
else:
    top_nodes = []

update_results("forensics", {"top_suspicious_nodes": top_nodes})
add_status_note("Graph-based suspicious node ranking completed.")
save_results_json()

print("Suspicious node ranking completed.")
print(f"  Ranked nodes        : {len(node_ranking)}")
print("\nTop 10 suspicious nodes:")
print(pd.DataFrame(top_nodes).head(10).to_string(index=False))


Suspicious node ranking completed.
  Ranked nodes        : 505

Top 10 suspicious nodes:
                                  node  out_suspicion  in_suspicion  out_degree  in_degree  pagerank  risk_score
                          192.168.1.31      54.127597      0.789583          65          1  0.002589   28.488431
                          192.168.1.30      44.671262      0.030391          57          1  0.001833   23.677989
                          192.168.1.34      44.681721      0.000000          51          0  0.001776   22.973044
                         192.168.1.190       9.772385      2.841106         158         14  0.008070   21.962900
                         192.168.1.195      16.019159      4.234221          38         11  0.003681   12.578666
2405:6e00:10ce:2c00:20c:29ff:feee:e07a       0.537619      0.000000         108          0  0.001776   11.015403
                         192.168.1.152       5.019691      5.229527          51         12  0.003442    9.877423
       

In [16]:
# =============================================================================
# SECTION 15: COUNTERFACTUAL CYBER DEFENSE (DiCE)
# =============================================================================

import dice_ml
from dice_ml import Dice

counterfactual_summary = {"status": "not_run", "details": []}

try:
    X_train_for_dice = X_train_raw.copy()
    X_test_for_dice  = X_test_raw.copy()

    # Use numeric labels so DiCE output aligns with predict_proba columns
    train_for_dice = X_train_for_dice.copy()
    train_for_dice[LABEL_COLUMN] = y_train

    continuous_features = [
        c for c in X_train_for_dice.columns
        if pd.api.types.is_numeric_dtype(X_train_for_dice[c])
    ]

    data_dice  = dice_ml.Data(
        dataframe=train_for_dice,
        continuous_features=continuous_features,
        outcome_name=LABEL_COLUMN,
    )
    model_dice = dice_ml.Model(model=model, backend="sklearn")
    exp_dice   = Dice(data_dice, model_dice, method="random")

    # Select most suspicious non-normal instance as query
    all_classes       = list(label_encoder.classes_)
    target_class_name = "normal" if "normal" in all_classes else all_classes[0]
    target_class_idx  = int(list(all_classes).index(target_class_name))

    non_normal_mask = forensic_table["predicted_label"] != target_class_name
    candidate_idx = (
        int(forensic_table.loc[non_normal_mask, "risk_vs_normal"].idxmax())
        if non_normal_mask.any()
        else int(forensic_table["risk_vs_normal"].idxmax())
    )

    query_instance      = X_test_for_dice.iloc[[candidate_idx]].copy().astype(object)
    original_prediction = str(label_encoder.classes_[int(y_pred[candidate_idx])])

    print("Counterfactual analysis started.")
    print(f"  Query index         : {candidate_idx}")
    print(f"  Original prediction : {original_prediction}")
    print(f"  Target class        : {target_class_name} (index {target_class_idx})")
    print(f"  Continuous features : {len(continuous_features)}")

    # Pass numeric index as desired_class — this is the key fix
    cf    = exp_dice.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_class=target_class_idx,
        features_to_vary="all",
    )
    cf_df = cf.cf_examples_list[0].final_cfs_df

    if cf_df is not None and not cf_df.empty:
        counterfactual_examples = cf_df.to_dict(orient="records")
        status_value = "success"
        print("  Counterfactual generated successfully.")
    else:
        counterfactual_examples = []
        status_value = "no_counterfactual_found"
        print("  No counterfactual found within constraints.")

    counterfactual_summary = {
        "status": status_value,
        "query_index": int(X_test_for_dice.index[candidate_idx]),
        "original_prediction": original_prediction,
        "target_class": target_class_name,
        "available_classes": all_classes,
        "counterfactual_examples": counterfactual_examples,
    }

except Exception as e:
    counterfactual_summary = {"status": "failed_or_skipped", "reason": str(e)}
    print(f"Counterfactual analysis failed: {e}")

update_results("counterfactuals", counterfactual_summary)
add_status_note(f"Counterfactual stage completed with status: {counterfactual_summary['status']}.")
save_results_json()

print(f"\nCounterfactual status: {counterfactual_summary['status']}")

Counterfactual analysis started.
  Query index         : 0
  Original prediction : ransomware
  Target class        : normal (index 5)
  Continuous features : 16


100%|██████████| 1/1 [00:00<00:00,  1.03it/s]

  Counterfactual generated successfully.

Counterfactual status: success


In [17]:
# =============================================================================
# SECTION 16: AUTOMATED EXPERIMENT VERDICT
# =============================================================================

f1_macro_val       = experiment_results.get("evaluation", {}).get("f1_macro", None)
roc_auc_val        = experiment_results.get("evaluation", {}).get("roc_auc_macro_ovr", None)
top_global_feats   = experiment_results.get("global_explainability", {}).get("top_features", [])
graph_n_nodes      = experiment_results.get("forensics", {}).get("graph_n_nodes", 0)
cf_status          = experiment_results.get("counterfactuals", {}).get("status", "not_run")

verdict = {"baseline_strength": "undetermined", "reasons": []}

if f1_macro_val is not None:
    if f1_macro_val >= 0.95:
        verdict["reasons"].append("Excellent macro-F1.")
    elif f1_macro_val >= 0.90:
        verdict["reasons"].append("Strong macro-F1.")
    else:
        verdict["reasons"].append("Macro-F1 requires improvement.")

if roc_auc_val is not None:
    if roc_auc_val >= 0.95:
        verdict["reasons"].append("Excellent probabilistic class separability.")
    elif roc_auc_val >= 0.90:
        verdict["reasons"].append("Promising ROC-AUC.")
    else:
        verdict["reasons"].append("ROC-AUC requires improvement.")

if len(top_global_feats) >= 10:
    verdict["reasons"].append("Global explainability evidence is available.")
else:
    verdict["reasons"].append("Global explainability evidence is limited.")

if graph_n_nodes and graph_n_nodes > 0:
    verdict["reasons"].append("Forensic graph layer is active.")
else:
    verdict["reasons"].append("Forensic graph layer is not active.")

if cf_status == "success":
    verdict["reasons"].append("Counterfactual defense evidence is available.")
elif cf_status == "failed_or_skipped":
    verdict["reasons"].append("Counterfactual stage still needs completion.")

positive_count = sum(
    any(kw in r for kw in ["Excellent", "Strong", "Promising", "available", "active"])
    for r in verdict["reasons"]
)

verdict["baseline_strength"] = (
    "strong" if positive_count >= 4 else
    "moderate" if positive_count >= 2 else
    "weak"
)

update_results("evaluation", {"automated_verdict": verdict})
save_results_json()

print("Automated verdict:")
print(json.dumps(verdict, indent=2))


Automated verdict:
{
  "baseline_strength": "strong",
  "reasons": [
    "Excellent macro-F1.",
    "Excellent probabilistic class separability.",
    "Global explainability evidence is available.",
    "Forensic graph layer is active.",
    "Counterfactual defense evidence is available."
  ]
}


In [18]:
# =============================================================================
# SECTION 17: ARTICLE ASSET GENERATION
# =============================================================================

generated_paths = []

# Per-class classification report (CSV and Excel)
report_df  = (
    pd.DataFrame(report_dict)
    .transpose()
    .reset_index()
    .rename(columns={"index": "class_or_metric"})
)
report_csv = register_artifact(
    save_dataframe(report_df, "classification_report.csv", index=False),
    "Per-class classification report (CSV)",
)
generated_paths.append(report_csv)

try:
    report_xlsx = register_artifact(
        save_excel(report_df, "classification_report.xlsx", index=False),
        "Per-class classification report (Excel)",
    )
    generated_paths.append(report_xlsx)
except Exception as e:
    print(f"Excel export skipped: {e}")

# Confusion matrix table
cm_df    = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
cm_csv   = register_artifact(
    save_dataframe(
        cm_df.reset_index().rename(columns={"index": "true_pred"}),
        "confusion_matrix_table.csv",
        index=False,
    ),
    "Confusion matrix table (CSV)",
)
generated_paths.append(cm_csv)

# Confusion matrix figure
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(label_encoder.classes_))
plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha="right")
plt.yticks(tick_marks, label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
thresh = cm.max() / 2.0 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
        )
plt.tight_layout()
cm_fig_path = FIGURES_DIR / "confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.close()
register_artifact(cm_fig_path, "Confusion matrix figure")
generated_paths.append(cm_fig_path)

# Global feature importance table and figure
global_xai   = experiment_results.get("global_explainability", {})
top_feats    = global_xai.get("top_features", [])

if top_feats:
    xai_df  = pd.DataFrame(top_feats)
    xai_csv = register_artifact(
        save_dataframe(xai_df, "global_feature_importance.csv", index=False),
        "Global feature importance table",
    )
    generated_paths.append(xai_csv)

    plot_df     = xai_df.head(15).copy()
    value_col   = "importance" if "importance" in plot_df.columns else plot_df.columns[-1]
    feature_col = "feature"    if "feature"    in plot_df.columns else plot_df.columns[0]

    plt.figure(figsize=(9, 6))
    plt.barh(
        plot_df[feature_col].astype(str)[::-1],
        plot_df[value_col][::-1],
    )
    plt.title("Top Global Features (SHAP)")
    plt.xlabel("Mean Absolute Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    xai_fig_path = FIGURES_DIR / "global_feature_importance.png"
    plt.savefig(xai_fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    register_artifact(xai_fig_path, "Global feature importance figure")
    generated_paths.append(xai_fig_path)

# Top suspicious nodes table
top_nodes_saved = experiment_results.get("forensics", {}).get("top_suspicious_nodes", [])
if top_nodes_saved:
    nodes_csv = register_artifact(
        save_dataframe(pd.DataFrame(top_nodes_saved), "top_suspicious_nodes.csv", index=False),
        "Top suspicious nodes table",
    )
    generated_paths.append(nodes_csv)

# Top suspicious flows table
if "forensic_table" in globals() and isinstance(forensic_table, pd.DataFrame) and not forensic_table.empty:
    suspicious_flows_df = (
        forensic_table
        .sort_values("suspicion_score", ascending=False)
        .head(100)
        .copy()
    )
    suspicious_csv = register_artifact(
        save_dataframe(suspicious_flows_df, "top_suspicious_flows.csv", index=False),
        "Top suspicious flows table",
    )
    generated_paths.append(suspicious_csv)

print("Article assets generated:")
for p in generated_paths:
    print(f"  - {p}")


Article assets generated:
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/classification_report.csv
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/classification_report.xlsx
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/confusion_matrix_table.csv
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/figures/confusion_matrix.png
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/global_feature_importance.csv
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/figures/global_feature_importance.png
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/top_suspicious_nodes.csv
  - /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/top_suspicious_flows.csv


In [19]:
# =============================================================================
# SECTION 18: ABLATION STUDY — GRAPH-AWARE RANKING EVALUATION
# =============================================================================

node_flow_counts  = {}
node_attack_counts = {}

for _, row in forensic_table.iterrows():
    src = row.get(SRC_IP_COL)
    if pd.isna(src):
        continue
    node_flow_counts[src]   = node_flow_counts.get(src, 0) + 1
    if row["predicted_label"] != "normal":
        node_attack_counts[src] = node_attack_counts.get(src, 0) + 1

malicious_nodes = set(
    node for node in node_flow_counts
    if node_attack_counts.get(node, 0) / node_flow_counts[node] > 0.5
)

print(f"Ground-truth malicious nodes : {len(malicious_nodes)}")
print(f"Total graph nodes            : {G.number_of_nodes()}")

def rank_full_graph(G, pagerank_scores):
    ranks = []
    for node in G.nodes():
        out_s  = sum(G[node][nbr].get("suspicion_avg", 0.0) for nbr in G.successors(node))
        in_s   = sum(G[pred][node].get("suspicion_avg", 0.0) for pred in G.predecessors(node))
        score  = (
            0.4 * out_s
            + 0.3 * in_s
            + 0.2 * pagerank_scores.get(node, 0.0)
            + 0.1 * (G.out_degree(node) + G.in_degree(node))
        )
        ranks.append((node, score))
    return [n for n, _ in sorted(ranks, key=lambda x: x[1], reverse=True)]

def rank_out_degree(G):
    return sorted(G.nodes(), key=lambda n: G.out_degree(n), reverse=True)

def rank_in_degree(G):
    return sorted(G.nodes(), key=lambda n: G.in_degree(n), reverse=True)

def rank_no_graph(G):
    scores = {}
    for node in G.nodes():
        out_s = sum(G[node][nbr].get("suspicion_sum", 0.0) for nbr in G.successors(node))
        in_s  = sum(G[pred][node].get("suspicion_sum", 0.0) for pred in G.predecessors(node))
        scores[node] = out_s + in_s
    return [n for n, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def precision_at_k(ranked, malicious, k):
    top_k = ranked[:k]
    return len([n for n in top_k if n in malicious]) / k if k > 0 else 0.0

def recall_at_k(ranked, malicious, k):
    top_k = ranked[:k]
    return len([n for n in top_k if n in malicious]) / len(malicious) if malicious else 0.0

def mean_reciprocal_rank(ranked, malicious):
    rr_sum = 0.0
    for node in malicious:
        if node in ranked:
            rr_sum += 1.0 / (ranked.index(node) + 1)
    return rr_sum / len(malicious) if malicious else 0.0

pagerank_scores = (
    nx.pagerank(G, alpha=0.85)
    if G.number_of_edges() > 0
    else {n: 0.0 for n in G.nodes()}
)

variants = {
    "Full Graph-Aware"  : rank_full_graph(G, pagerank_scores),
    "Out-Degree Only"   : rank_out_degree(G),
    "In-Degree Only"    : rank_in_degree(G),
    "No-Graph Baseline" : rank_no_graph(G),
}

k_values       = [3, 5, 10]
ablation_results = {}

for name, ranked in variants.items():
    ablation_results[name] = {
        "precision_at_k": {k: precision_at_k(ranked, malicious_nodes, k) for k in k_values},
        "recall_at_10"  : recall_at_k(ranked, malicious_nodes, 10),
        "mrr"           : mean_reciprocal_rank(ranked, malicious_nodes),
    }

x      = np.arange(len(k_values))
width  = 0.2
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, results) in enumerate(ablation_results.items()):
    prec_vals = [results["precision_at_k"][k] for k in k_values]
    ax.bar(x + i * width, prec_vals, width, label=name, color=colors[i])

ax.set_xlabel("Top-k Threshold")
ax.set_ylabel("Precision@k")
ax.set_title("Ablation Study: Precision at Multiple Ranking Thresholds")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f"@{k}" for k in k_values])
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

fig4_path = FIGURES_DIR / "ablation_precision_at_k.png"
plt.savefig(fig4_path, dpi=300, bbox_inches="tight")
plt.close()
register_artifact(fig4_path, "Ablation study Precision@k figure")

names       = list(ablation_results.keys())
recall_vals = [ablation_results[n]["recall_at_10"] for n in names]
mrr_vals    = [ablation_results[n]["mrr"] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].bar(names, recall_vals, color=colors)
axes[0].set_title("Recall@10 by Ranking Variant")
axes[0].set_ylabel("Recall@10")
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

axes[1].bar(names, mrr_vals, color=colors)
axes[1].set_title("Mean Reciprocal Rank (MRR) by Ranking Variant")
axes[1].set_ylabel("MRR")
axes[1].set_ylim(0, max(mrr_vals) * 1.3 if max(mrr_vals) > 0 else 1.0)
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Ablation Study: Ranking Quality Beyond Coverage", fontsize=13)
plt.tight_layout()

fig5_path = FIGURES_DIR / "ablation_recall_mrr.png"
plt.savefig(fig5_path, dpi=300, bbox_inches="tight")
plt.close()
register_artifact(fig5_path, "Ablation study Recall@10 and MRR figure")

ablation_summary = {
    variant: {
        "precision_at_3" : results["precision_at_k"][3],
        "precision_at_5" : results["precision_at_k"][5],
        "precision_at_10": results["precision_at_k"][10],
        "recall_at_10"   : results["recall_at_10"],
        "mrr"            : results["mrr"],
    }
    for variant, results in ablation_results.items()
}

update_results("forensics", {"ablation_study": ablation_summary})
add_status_note("Ablation study completed.")
save_results_json()

print("Ablation study completed.")
for name, res in ablation_summary.items():
    print(f"  {name}")
    for metric, val in res.items():
        print(f"    {metric:20s}: {val:.4f}")

Ground-truth malicious nodes : 13
Total graph nodes            : 505
Ablation study completed.
  Full Graph-Aware
    precision_at_3      : 1.0000
    precision_at_5      : 0.6000
    precision_at_10     : 0.5000
    recall_at_10        : 0.3846
    mrr                 : 0.1854
  Out-Degree Only
    precision_at_3      : 0.3333
    precision_at_5      : 0.4000
    precision_at_10     : 0.5000
    recall_at_10        : 0.3846
    mrr                 : 0.1047
  In-Degree Only
    precision_at_3      : 0.0000
    precision_at_5      : 0.0000
    precision_at_10     : 0.1000
    recall_at_10        : 0.0769
    mrr                 : 0.0186
  No-Graph Baseline
    precision_at_3      : 1.0000
    precision_at_5      : 0.8000
    precision_at_10     : 0.6000
    recall_at_10        : 0.4615
    mrr                 : 0.1967


In [20]:
# =============================================================================
# SECTION 19: FORENSIC FLOW CONCENTRATION ANALYSIS
# =============================================================================

if graph_available and G.number_of_edges() > 0:
    edge_scores = sorted(
        [data.get("suspicion_avg", 0.0) for _, _, data in G.edges(data=True)],
        reverse=True,
    )

    total_suspicion = sum(edge_scores)
    cumulative      = (
        np.cumsum(edge_scores) / total_suspicion
        if total_suspicion > 0
        else np.zeros(len(edge_scores))
    )
    x_axis = np.arange(1, len(edge_scores) + 1) / len(edge_scores)

    plt.figure(figsize=(9, 5))
    plt.plot(x_axis * 100, cumulative * 100, color="#2196F3", linewidth=2)
    plt.xlabel("Percentage of Ranked Flows (%)")
    plt.ylabel("Cumulative Suspicious Activity (%)")
    plt.title("Cumulative Concentration of Suspicious Flows Ranked by Risk Score")
    plt.grid(linestyle="--", alpha=0.5)
    plt.tight_layout()

    fig6_path = FIGURES_DIR / "flow_concentration.png"
    plt.savefig(fig6_path, dpi=300, bbox_inches="tight")
    plt.close()
    register_artifact(fig6_path, "Suspicious flow concentration curve")

    top10_pct = float(cumulative[max(0, len(edge_scores) // 10 - 1)] * 100)
    top20_pct = float(cumulative[max(0, len(edge_scores) // 5  - 1)] * 100)

    concentration_summary = {
        "total_edges": len(edge_scores),
        "top_10_pct_flows_account_for_pct_suspicion": round(top10_pct, 2),
        "top_20_pct_flows_account_for_pct_suspicion": round(top20_pct, 2),
    }

    update_results("forensics", {"flow_concentration": concentration_summary})
    add_status_note("Forensic flow concentration analysis completed.")
    save_results_json()

    print("Flow concentration analysis completed.")
    print(f"  Total edges      : {len(edge_scores)}")
    print(f"  Top 10% flows    : {top10_pct:.1f}% of total suspicion")
    print(f"  Top 20% flows    : {top20_pct:.1f}% of total suspicion")
    print(f"  Figure saved     : {fig6_path}")
else:
    print("Graph not available. Flow concentration analysis skipped.")

Flow concentration analysis completed.
  Total edges      : 667
  Top 10% flows    : 33.9% of total suspicion
  Top 20% flows    : 61.7% of total suspicion
  Figure saved     : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/figures/flow_concentration.png


In [21]:
# =============================================================================
# SECTION 20: COUNTERFACTUAL PROBABILITY SHIFT VISUALIZATION
# =============================================================================

cf_status   = experiment_results.get("counterfactuals", {}).get("status", "not_run")
cf_examples = experiment_results.get("counterfactuals", {}).get("counterfactual_examples", [])
query_idx   = experiment_results.get("counterfactuals", {}).get("query_index", None)

if cf_status == "success" and cf_examples and query_idx is not None:
    class_names = label_encoder.classes_

    query_row  = (
        X_test_raw.loc[[query_idx]]
        if query_idx in X_test_raw.index
        else X_test_raw.iloc[[0]]
    )
    orig_proba = model.predict_proba(query_row)[0]

    cf_record = cf_examples[0]
    cf_row    = query_row.copy()
    for col, val in cf_record.items():
        if col in cf_row.columns:
            cf_row[col] = val
    cf_proba = model.predict_proba(cf_row)[0]

    x     = np.arange(len(class_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width / 2, orig_proba, width, label="Original Instance", color="#F44336", alpha=0.85)
    ax.bar(x + width / 2, cf_proba,   width, label="Counterfactual",    color="#4CAF50", alpha=0.85)

    ax.set_xlabel("Class")
    ax.set_ylabel("Predicted Probability")
    ax.set_title("Counterfactual Prediction Shift for a Representative Attack Instance")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()

    fig7_path = FIGURES_DIR / "counterfactual_shift.png"
    plt.savefig(fig7_path, dpi=300, bbox_inches="tight")
    plt.close()
    register_artifact(fig7_path, "Counterfactual probability shift figure")

    normal_idx = list(class_names).index("normal")
    cf_viz_summary = {
        "normal_prob_original"      : float(orig_proba[normal_idx]),
        "normal_prob_counterfactual": float(cf_proba[normal_idx]),
        "original_probabilities"    : {c: float(p) for c, p in zip(class_names, orig_proba)},
        "counterfactual_probabilities": {c: float(p) for c, p in zip(class_names, cf_proba)},
    }

    update_results("counterfactuals", {"probability_shift_viz": cf_viz_summary})
    add_status_note("Counterfactual probability shift visualization completed.")
    save_results_json()

    print("Counterfactual probability shift visualization completed.")
    print(f"  Normal prob (original)      : {cf_viz_summary['normal_prob_original']:.4f}")
    print(f"  Normal prob (counterfactual): {cf_viz_summary['normal_prob_counterfactual']:.4f}")
    print(f"  Figure saved                : {fig7_path}")
else:
    print(f"Counterfactual status is '{cf_status}'. Visualization skipped.")

Counterfactual probability shift visualization completed.
  Normal prob (original)      : 0.0000
  Normal prob (counterfactual): 0.4250
  Figure saved                : /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/figures/counterfactual_shift.png


In [22]:
# =============================================================================
# SECTION 21: COMPACT RESULT REPORT AND FINAL SAVE
# =============================================================================

eval_res     = experiment_results.get("evaluation", {})
model_res    = experiment_results.get("model", {})
forensic_res = experiment_results.get("forensics", {})
cf_res       = experiment_results.get("counterfactuals", {})
xai_res      = experiment_results.get("global_explainability", {})

# Save structured summary stats for MethodsX reproducibility
summary_stats = {
    "model_name"            : model_res.get("model_name", "N/A"),
    "n_estimators"          : model_res.get("hyperparameters", {}).get("n_estimators", "N/A"),
    "training_time_sec"     : round(model_res.get("training_time_sec", 0), 2),
    "training_accuracy"     : round(model_res.get("training_accuracy", 0), 4),
    "test_accuracy"         : round(eval_res.get("accuracy", 0), 4),
    "macro_precision"       : round(eval_res.get("precision_macro", 0), 4),
    "macro_recall"          : round(eval_res.get("recall_macro", 0), 4),
    "macro_f1"              : round(eval_res.get("f1_macro", 0), 4),
    "roc_auc_macro_ovr"     : round(eval_res.get("roc_auc_macro_ovr", 0), 4),
    "n_transformed_features": model_res.get("n_transformed_features", "N/A"),
    "graph_available"       : forensic_res.get("graph_available", False),
    "graph_n_nodes"         : forensic_res.get("graph_n_nodes", 0),
    "graph_n_edges"         : forensic_res.get("graph_n_edges", 0),
    "xai_method"            : xai_res.get("method_selected", "N/A"),
    "counterfactual_status" : cf_res.get("status", "N/A"),
}

stats_path = INTERMEDIATE_DIR / "experiment_summary_stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(summary_stats, f, indent=2, ensure_ascii=False)

# Build compact human-readable report
summary_lines = [
    "Forensic-Ready Explainable IDS — Compact Result Report",
    "=" * 58,
    f"Run ID             : {experiment_results.get('run_id', 'N/A')}",
    f"Created at         : {experiment_results.get('created_at', 'N/A')}",
    "",
    "Model:",
    f"  Model name        : {summary_stats['model_name']}",
    f"  Trees             : {summary_stats['n_estimators']}",
    f"  Training time (s) : {summary_stats['training_time_sec']}",
    f"  Training accuracy : {summary_stats['training_accuracy']}",
    "",
    "Core predictive metrics:",
    f"  Test accuracy     : {summary_stats['test_accuracy']}",
    f"  Macro precision   : {summary_stats['macro_precision']}",
    f"  Macro recall      : {summary_stats['macro_recall']}",
    f"  Macro F1          : {summary_stats['macro_f1']}",
    f"  ROC-AUC macro OVR : {summary_stats['roc_auc_macro_ovr']}",
    "",
    "Explainability:",
    f"  XAI method        : {summary_stats['xai_method']}",
    f"  Transformed feats : {summary_stats['n_transformed_features']}",
    "",
    "Forensic graph:",
    f"  Graph available   : {summary_stats['graph_available']}",
    f"  Nodes             : {summary_stats['graph_n_nodes']}",
    f"  Edges             : {summary_stats['graph_n_edges']}",
    "",
    "Counterfactual:",
    f"  Status            : {summary_stats['counterfactual_status']}",
    "",
    "Saved artifacts:",
]

for item in SAVED_ARTIFACTS:
    summary_lines.append(f"  - {item['description']}: {item['path']}")

summary_text        = "\n".join(summary_lines)
summary_report_path = register_artifact(
    save_text_report("summary_report.txt", summary_text),
    "Compact summary report",
)

update_results("artifact_registry", SAVED_ARTIFACTS)
save_results_json()

print(summary_text)
print("\nIntermediate files:")
print(f"  JSON  : {RESULTS_JSON}")
print(f"  CSV   : {RESULTS_CSV}")
print(f"  Stats : {stats_path}")
print("\nSummary report saved to:")
print(f"  {summary_report_path}")

Forensic-Ready Explainable IDS — Compact Result Report
Run ID             : 20260331_065437
Created at         : 2026-03-31T06:54:37.778204

Model:
  Model name        : RandomForestClassifier
  Trees             : 200
  Training time (s) : 213.55
  Training accuracy : 0.9999

Core predictive metrics:
  Test accuracy     : 0.989
  Macro precision   : 0.9642
  Macro recall      : 0.974
  Macro F1          : 0.9687
  ROC-AUC macro OVR : 0.9979

Explainability:
  XAI method        : shap
  Transformed feats : 887

Forensic graph:
  Graph available   : True
  Nodes             : 505
  Edges             : 667

Counterfactual:
  Status            : success

Saved artifacts:
  - Per-class classification report (CSV): /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/classification_report.csv
  - Per-class classification report (Excel): /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/article_assets/tables/classification_report.xlsx
  - Confusion matrix table (CS